In [28]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.metrics import adjusted_rand_score

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [29]:
nuc_color = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
cyto_color = ["#F56867", "#FEB915", "#C798EE", "#59BE86", "#7495D3", "#6D1A9C", "#15821E", "#3A84E6", "#997273", "#D1D1D1"]

sample = "Xenium_5K_BC"

In [30]:
# paths
intermediate_path = f"../../data/{sample}/intermediate_data/"
processed_path = f"../../data/{sample}/processed_data/"
output_path = f"../../output/1_exploration/{sample}/"

# read data
adata = sc.read_h5ad(intermediate_path + "adata.h5ad")
cell_ids = np.load(processed_path + "cell_ids.npy", allow_pickle=True)
assert adata.shape[0] > cell_ids.shape[0], "cell_ids should be a subset of adata.obs_names"
adata_tumor = adata[adata.obs["cell_id"].isin(cell_ids)].copy()

adata_tumor_nuc = sc.read_h5ad(output_path + "adata_nuclear_embedded.h5ad")
adata_tumor_cyto = sc.read_h5ad(output_path + "adata_cytoplasmic_embedded.h5ad")

assert adata_tumor.shape[0] == adata_tumor_nuc.shape[0], "adata_tumor and adata_tumor_nuc should have the same number of cells"
assert adata_tumor_nuc.shape[0] == adata_tumor_cyto.shape[0], "Nuclear and cytoplasmic embeddings should have the same number of cells"

In [31]:
# Clustering
res_nuc, res_cyto = 0.6, 1.0
sc.tl.louvain(adata_tumor_nuc, resolution = res_nuc, key_added = "subtype_louvain")
sc.tl.louvain(adata_tumor_cyto, resolution = res_cyto, key_added = "subtype_louvain")

adata_tumor.obs["subtype_louvain_nuc"] = adata_tumor_nuc.obs["subtype_louvain"].values
adata_tumor.obs["subtype_louvain_cyto"] = adata_tumor_cyto.obs["subtype_louvain"].values

print(f"Number of clusters (nuclear): {adata_tumor.obs['subtype_louvain_nuc'].nunique()}")
print(f"Number of clusters (cytoplasmic): {adata_tumor.obs['subtype_louvain_cyto'].nunique()}")

Number of clusters (nuclear): 6
Number of clusters (cytoplasmic): 6


In [32]:
# UMAP plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata_tumor_nuc, color="subtype_louvain", ax=axes[0], show=False, title="Nuclear expression")
sc.pl.umap(adata_tumor_cyto, color="subtype_louvain", ax=axes[1], show=False, title="Cytoplasmic expression")
plt.savefig(output_path + "umap_clustering.png", dpi=300, bbox_inches="tight")
plt.close()

In [33]:
# ARI
df = pd.DataFrame({"subtype_louvain_nuc": adata_tumor.obs["subtype_louvain_nuc"], "subtype_louvain_cyto": adata_tumor.obs["subtype_louvain_cyto"]})
df.to_csv(output_path + "clustering_results.csv", index=False)

ari = adjusted_rand_score(adata_tumor.obs["subtype_louvain_nuc"], adata_tumor.obs["subtype_louvain_cyto"])
print(f"Adjusted Rand Index (ARI) between nuclear and cytoplasmic clustering: {ari:.4f}")

Adjusted Rand Index (ARI) between nuclear and cytoplasmic clustering: 0.0856


In [34]:
# Scatter plot
for label, palette in zip(["subtype_louvain_nuc", "subtype_louvain_cyto"], [nuc_color, cyto_color]):
    sc.set_figure_params(figsize = (7, 12))
    ax = sc.pl.scatter(adata_tumor, x = "global_x", y = "global_y", color = label, palette = palette, size = 2, alpha = 1, show = False)
    ax.grid(False)
    plt.savefig(output_path + f"scatter_{label}.png", dpi = 300, bbox_inches = "tight")
    plt.close()

In [35]:
# Within-compartment DE analysis
for adata_tmp, label in zip([adata_tumor_nuc, adata_tumor_cyto], ["nuclear", "cytoplasmic"]):
    
    sc.tl.rank_genes_groups(adata_tmp, "subtype_louvain", method="wilcoxon")
    
    names = adata_tmp.uns["rank_genes_groups"]["names"]
    names = pd.DataFrame(names)
    logfc = adata_tmp.uns["rank_genes_groups"]["logfoldchanges"]
    logfc = pd.DataFrame(logfc)
    pvals = adata_tmp.uns["rank_genes_groups"]["pvals"]
    pvals = pd.DataFrame(pvals)

    for i in adata_tmp.obs["subtype_louvain"].unique():
        df = {"names": names[i], "logfc": logfc[i], "pvals": pvals[i]}
        df = pd.DataFrame(df)
        df = df[df["logfc"] >= 0]
        df = df[df["pvals"] <= 0.05]
        df = df.sort_values(by = ["logfc"], ascending = False)
        file = output_path + f"within_compartment_{label}_cluster_{i}.csv"
        df.to_csv(file, index = False)

In [36]:
# Cross compartments DE analysis
adata_tumor_cyto.obs["subtype_louvain_nuc"] = adata_tumor_nuc.obs["subtype_louvain"].values

sc.tl.rank_genes_groups(adata_tumor_cyto, "subtype_louvain_nuc", method="wilcoxon")
    
names = adata_tumor_cyto.uns["rank_genes_groups"]["names"]
names = pd.DataFrame(names)
logfc = adata_tumor_cyto.uns["rank_genes_groups"]["logfoldchanges"]
logfc = pd.DataFrame(logfc)
pvals = adata_tumor_cyto.uns["rank_genes_groups"]["pvals"]
pvals = pd.DataFrame(pvals)

for i in adata_tumor_cyto.obs["subtype_louvain_nuc"].unique():
    df = {"names": names[i], "logfc": logfc[i], "pvals": pvals[i]}
    df = pd.DataFrame(df)
    df = df[df["logfc"] >= 0]
    df = df[df["pvals"] <= 0.05]
    df = df.sort_values(by = ["logfc"], ascending = False)
    file = output_path + f"cross_compartment_{label}_cluster_{i}.csv"
    df.to_csv(file, index = False)

In [ ]:
# Within-compartment heatmap